In [1]:
# ==============================================================================
# FULL SSD300 (VGG16) TRAINING PIPELINE FOR PASCAL VOC 2012
# ==============================================================================
!pip install torchmetrics -q

import os, time, json, torch, torchvision
import torch.nn as nn
import torchvision.transforms.functional as F
from torch.utils.data import Dataset, DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("✅ Cell 1: Setup and Imports Complete")

✅ Cell 1: Setup and Imports Complete


In [2]:
# 1. CONFIGURATION
CONFIG = {
    "num_classes": 21, 
    "batch_size": 16, 
    "epochs": 50, 
    "lr": 0.0005,        # Lowered to prevent NaN explosion
    "momentum": 0.9, 
    "weight_decay": 5e-4, 
    "patience": 10, 
    "step_size": 15, 
    "gamma": 0.1,
    "max_norm": 2.0,     # New: Prevents gradients from exploding
    "output_dir": "/kaggle/working/", 
    "best_model_path": "/kaggle/working/best.pth",
}

VOC_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat", 
    "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person", "pottedplant", 
    "sheep", "sofa", "train", "tvmonitor"
]
VOC_CLASS_TO_IDX = {cls_name: idx for idx, cls_name in enumerate(VOC_CLASSES)}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("✅ Cell 2: Configuration Complete")

✅ Cell 2: Configuration Complete


In [3]:
# 2. BULLETPROOF DATASET (WITH BOX CLAMPING FIX)
class VOCDetectionDataset(Dataset):
    def __init__(self, root, image_set='train'):
        self.voc = torchvision.datasets.VOCDetection(root, year='2012', image_set=image_set, download=True)
        
    def __getitem__(self, idx):
        img, target_dict = self.voc[idx]
        w, h = img.size # Get dimensions before converting to tensor
        
        boxes, labels = [], []
        objects = target_dict['annotation']['object']
        if not isinstance(objects, list): objects = [objects] 
        
        for obj in objects:
            bbox = obj['bndbox']
            
            # CLAMPING FIX: Force coordinates to stay within [0, width] and [0, height]
            xmin = max(0.0, float(bbox['xmin']) - 1.0)
            ymin = max(0.0, float(bbox['ymin']) - 1.0)
            xmax = min(float(w), float(bbox['xmax']) - 1.0)
            ymax = min(float(h), float(bbox['ymax']) - 1.0)
            
            # Only append valid boxes (prevents 0-area boxes)
            if xmax > xmin and ymax > ymin:
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(VOC_CLASS_TO_IDX[obj['name']])
            
        img = F.to_tensor(img.convert("RGB"))
        if img.shape[0] != 3:
            if img.shape[0] == 1: img = img.repeat(3, 1, 1)
            elif img.shape[0] == 4: img = img[:3, :, :]
                
        target = {"boxes": torch.as_tensor(boxes, dtype=torch.float32), "labels": torch.as_tensor(labels, dtype=torch.int64)}
        return img, target
        
    def __len__(self): return len(self.voc)

def collate_fn(batch): return tuple(zip(*batch))

print("Loading Data...")
train_loader = DataLoader(VOCDetectionDataset("/kaggle/working/data", "train"), batch_size=CONFIG["batch_size"], shuffle=True, num_workers=4, collate_fn=collate_fn)
val_loader = DataLoader(VOCDetectionDataset("/kaggle/working/data", "val"), batch_size=CONFIG["batch_size"], shuffle=False, num_workers=4, collate_fn=collate_fn)

print("✅ Cell 3: Dataset Preparation Complete")

Loading Data...


100%|██████████| 2.00G/2.00G [00:57<00:00, 34.9MB/s]


✅ Cell 3: Dataset Preparation Complete


In [4]:
# 3. MULTI-GPU FIX
class ObjectDetectionDataParallel(nn.DataParallel):
    def scatter(self, inputs, kwargs, device_ids):
        images, targets = inputs[0], inputs[1] if len(inputs) > 1 else None
        chunk_size = (len(images) + len(device_ids) - 1) // len(device_ids)
        scattered_inputs, scattered_kwargs = [], []
        for i, device_id in enumerate(device_ids):
            start, end = i * chunk_size, (i + 1) * chunk_size
            sub_images = images[start:end]
            if not sub_images: continue
            sub_images = [img.to(device_id) for img in sub_images]
            if targets is not None:
                sub_targets = [{k: v.to(device_id) for k, v in t.items()} for t in targets[start:end]]
                scattered_inputs.append((sub_images, sub_targets))
            else:
                scattered_inputs.append((sub_images,))
            scattered_kwargs.append(kwargs)
        return scattered_inputs, scattered_kwargs

# 4. INIT SSD MODEL (WITH WARMUP FIX)
print("Initializing SSD300 Model...")
CONFIG["lr"] = 0.002 # RESTORE STANDARD LR

model = torchvision.models.detection.ssd300_vgg16(weights_backbone='DEFAULT', num_classes=CONFIG["num_classes"]).to(device)
if torch.cuda.device_count() > 1: model = ObjectDetectionDataParallel(model)

optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=CONFIG["lr"], momentum=CONFIG["momentum"], weight_decay=CONFIG["weight_decay"])

# THE WARMUP SCHEDULER
warmup_iters = len(train_loader)
lr_lambda = lambda it: (it + 1) / warmup_iters if it < warmup_iters else 1.0
warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=CONFIG["step_size"], gamma=CONFIG["gamma"])

print("✅ Model Initialization Complete")

# 5. REVISED TRAINING LOOP
from tqdm import tqdm
print("Starting Training...")
best_map50 = -1.0  
epochs_no_improve = 0
history = {"train_loss": [], "val_map50": [], "val_map50_95": [], "precision": [], "recall": []}
start_time = time.time()

for epoch in range(CONFIG["epochs"]):
    model.train()
    epoch_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
    for images, targets in pbar:
        images = list(img.to(device) for img in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        loss_dict = model(images, targets)
        losses = sum(loss.mean() for loss in loss_dict.values())
        
        # NaN Safety Net
        if torch.isnan(losses):
            continue 
            
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG.get("max_norm", 2.0))
        optimizer.step()
        
        # STEP WARMUP ONLY ON FIRST EPOCH
        if epoch == 0:
            warmup_scheduler.step()
            
        epoch_loss += losses.item()
        pbar.set_postfix({"loss": f"{losses.item():.4f}"})
        
    scheduler.step()
    history["train_loss"].append(epoch_loss / len(train_loader))
    
    # Validation
    eval_model = model.module if hasattr(model, 'module') else model
    eval_model.eval()
    metric = MeanAveragePrecision(class_metrics=True) 
    
    with torch.no_grad():
        for images, targets in val_loader:
            outputs = eval_model(list(img.to(device) for img in images))
            metric.update([{k: v.cpu() for k, v in t.items()} for t in outputs], [{k: v.cpu() for k, v in t.items()} for t in targets])
            
    metrics = metric.compute()
    map50 = metrics['map_50'].item()
    history["val_map50"].append(map50)
    history["val_map50_95"].append(metrics['map'].item())
    history["precision"].append(metrics['map'].item())
    history["recall"].append(metrics['mar_100'].item())
    
    print(f"Epoch [{epoch+1}/{CONFIG['epochs']}] Loss: {epoch_loss/len(train_loader):.4f} | mAP@50: {map50:.4f}")
    
    if map50 > best_map50:
        best_map50 = map50
        epochs_no_improve = 0
        torch.save(eval_model.state_dict(), CONFIG["best_model_path"])
        print(f"   --> 🌟 Saved new best model with mAP: {best_map50:.4f}")
    else:
        epochs_no_improve += 1
        
    if epochs_no_improve >= CONFIG["patience"]:
        print("Early stopping triggered.")
        break

print("✅ Training complete. Weights are saved.")

Initializing SSD300 Model...
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 217MB/s]


✅ Model Initialization Complete
Starting Training...


Epoch [1/50] Loss: 33.6475 | mAP@50: 0.0004
   --> 🌟 Saved new best model with mAP: 0.0004


Epoch [2/50] Loss: 8.0846 | mAP@50: 0.0024
   --> 🌟 Saved new best model with mAP: 0.0024


...

Epoch [36/50] Loss: 6.6348 | mAP@50: 0.0099


Epoch [37/50] Loss: 6.6310 | mAP@50: 0.0098


Epoch [38/50] Loss: 6.6313 | mAP@50: 0.0097


Epoch [39/50] Loss: 6.6340 | mAP@50: 0.0098


Epoch [40/50] Loss: 6.6249 | mAP@50: 0.0099
Early stopping triggered.
✅ Training complete. Weights are saved.


In [5]:
# 6. EXTRACT METRICS & PLOT
print("Calculating FPS...")
eval_model.load_state_dict(torch.load(CONFIG["best_model_path"]))
eval_model.eval()
latencies = []
with torch.no_grad():
    for _ in range(10): eval_model([torch.rand(3, 300, 300).to(device)]) # Warmup
    for i, (images, _) in enumerate(val_loader):
        if i >= 100: break
        start = time.time()
        eval_model([images[0].to(device)])
        latencies.append((time.time() - start) * 1000)

avg_latency = sum(latencies) / len(latencies)
class_ap_dict = {VOC_CLASSES[i+1]: metrics['map_per_class'].tolist()[i] for i in range(20)}

with open("metrics.json", "w") as f:
    json.dump({"mAP_50": best_map50, "fps": 1000.0/avg_latency, "latency_ms": avg_latency, "train_time_min": (time.time()-start_time)/60, "per_class_ap": class_ap_dict}, f, indent=4)

plt.plot(history["train_loss"], label='Loss', color='red'); plt.legend(); plt.savefig("loss_curve.png"); plt.close()
plt.plot(history["val_map50"], label='mAP@50', color='blue'); plt.legend(); plt.savefig("map_curve.png"); plt.close()
!zip -r outputs.zip /kaggle/working/ -x "*/data/*" "*/.ipynb_checkpoints/*"
print("✅ ALL DONE! ZIP FILE CREATED.")

Calculating FPS...
  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/map_curve.png (deflated 10%)
  adding: kaggle/working/metrics.json (deflated 53%)
  adding: kaggle/working/best.pth (deflated 8%)
  adding: kaggle/working/loss_curve.png (deflated 21%)
  adding: kaggle/working/__notebook__.ipynb (deflated 96%)
✅ ALL DONE! ZIP FILE CREATED.
